In [7]:
import ast
import json
import os
import re
import subprocess
import textwrap
from pathlib import Path
from statistics import mean
from typing import Any, cast

import httpx
from anthropic import Anthropic, omit
from anthropic.types import Message, MessageParam, TextBlock
from dotenv import dotenv_values
from IPython.display import Markdown


def initialize_env() -> None:
    env_path = Path.cwd().parent / ".env.template"
    for key, ref in dotenv_values(env_path).items():
        if ref is None:
            continue
        if not os.environ.get(key):
            os.environ[key] = subprocess.run(
                ["op", "read", ref], capture_output=True, text=True, check=True
            ).stdout.strip()


def anthropic_client() -> Anthropic:
    return Anthropic(
        base_url="https://openrouter.ai/api",
        api_key=os.environ["OPENROUTER_API_KEY"],
    )


def model(name: str = "llama") -> str:
    models = {
        "deepseek": "deepseek/deepseek-v4.1-flash",
        "llama": "meta-llama/llama-3.1-8b-instruct",
        "llama-70b": "meta-llama/llama-3.3-70b-instruct",
        "llama-flagship": "meta-llama/llama-4-maverick",
        "mistral": "mistralai/mistral-small-3.1-24b-instruct",
        "haiku": "anthropic/claude-haiku-4.5",
        "qwen": "qwen/qwen-2.5-coder-32b-instruct",
    }
    return models.get(name, models["llama"])


initialize_env()
messages: list[MessageParam] = []

In [1]:
def get_reply(message: Message) -> str:
    text = next(
        (block.text for block in message.content if isinstance(block, TextBlock)),
        "",
    )
    if not text:
        return f"[no text block; stop_reason={message.stop_reason}, content={message.content!r}]"
    return text


def add_user_message(messages: list[MessageParam], text: str) -> None:
    user_message: MessageParam = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages: list[MessageParam], text: str) -> None:
    assistant_message: MessageParam = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def wrap_text(text: str, width: int = 120) -> str:
    return "\n".join(textwrap.fill(line, width=width) for line in text.splitlines())


def remaining_credits() -> None:
    response = httpx.get(
        "https://openrouter.ai/api/v1/credits",
        headers={"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}"},
        timeout=10.0,
    )
    response.raise_for_status()
    data = response.json()["data"]
    remaining = data["total_credits"] - data["total_usage"]
    display(Markdown(f"---\nRemaining: **${remaining:.2f}**"))

In [12]:
def chat(
    messages: list[MessageParam],
    system: str | None = None,
    temperature: float = 1.0,
    stop_sequences: list[str] | None = None,
) -> str:
    params: dict[str, Any] = {
        "model": model(),
        "max_tokens": 10000,
        "thinking": {"type": "disabled"},
        "messages": messages,
        "system": system or omit,
        "temperature": temperature,
        "stop_sequences": stop_sequences or omit,
    }
    message = cast(Message, anthropic_client().messages.create(**params))
    return get_reply(message)


def chat_stream(
    messages: list[MessageParam],
    system: str | None = None,
    temperature: float = 1.0,
    stop_sequences: list[str] | None = None,
):
    params: dict[str, Any] = {
        "model": model(),
        "max_tokens": 10000,
        "thinking": {"type": "disabled"},
        "messages": messages,
        "system": system or omit,
        "temperature": temperature,
        "stop_sequences": stop_sequences or omit,
    }
    with anthropic_client().messages.stream(**params) as stream:
        for text in stream.text_stream:
            print(wrap_text(f"{text}"), end="")
    return get_reply(stream.get_final_message())

In [ ]:
messages.clear()
add_user_message(messages, "Create a 1 sentence fake database description.")
Markdown(f"\n---\n{chat_stream(messages, temperature=0.999)}")

In [ ]:
messages.clear()

while True:
    user_input = input("You: ")
    if user_input == "/exit":
        break

    add_user_message(messages, user_input)
    print(wrap_text(f"You: {user_input}"))

    reply = chat(
        messages,
        system="You are a scientific assistant which popularizes complex scientific concepts.",
    )
    add_assistant_message(messages, reply)

    print(wrap_text(f"AI: {reply}"))
    print("\n---\n")

In [ ]:
messages.clear()
add_user_message(messages, "Generate AWS EventBridge rule as JSON")
add_assistant_message(messages, "```json\n")
print(wrap_text(chat(messages, stop_sequences=["```"]).strip()))
remaining_credits()

In [ ]:
messages.clear()
add_user_message(messages, "Generate 3 different short AWS CLI commands")
add_assistant_message(
    messages, "Here are three different short AWS CLI commands without comments:\n```bash\n"
)
print(chat(messages, stop_sequences=["```"]).strip())
remaining_credits()

In [17]:
def generate_dataset():
    prompt = textwrap.dedent("""
        Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
        that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
        each representing task that requires Python, JSON, or a Regex to complete.

        Example output:
        ```json
        [
            {
                "task": "Description of task",
                "format": "json" or "python" or "regex"
            },
            ...additional
        ]
        ```

        * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
        * Focus on tasks that do not require writing much code

        Please generate 3 objects.
    """).strip()

    messages.clear()
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json\n")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)


dataset = generate_dataset()
print(dataset)
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

[{'task': 'Write a Python function to validate if a string is a valid email address', 'format': 'python'}, {'task': 'Generate a JSON object to represent a product catalog with the product ID, name, and price', 'format': 'json'}, {'task': 'Create a regular expression to match any phone number in the format of (XXX) XXX-XXXX or XXX.XXX.XXXX', 'format': 'regex'}]


In [14]:
# Function to grade a test case + output using a model
def grade_by_model(test_case: dict[str, Any], output: str) -> dict[str, Any]:
    eval_prompt = textwrap.dedent(f"""
        You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

        Original Task:
        <task>
        {test_case["task"]}
        </task>

        Solution to Evaluate:
        <solution>
        {output}
        </solution>

        Output Format
        Provide your evaluation as a structured JSON object with the following fields, in this specific order:
        - "strengths": An array of 1-3 key strengths
        - "weaknesses": An array of 1-3 key areas for improvement
        - "reasoning": A concise explanation of your overall assessment
        - "score": A number between 1-10

        Respond with JSON. Keep your response concise and direct.
        Example response shape:
        {{
            "strengths": string[],
            "weaknesses": string[],
            "reasoning": string,
            "score": number
        }}
    """).strip()

    messages.clear()
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [ ]:
def validate_json(text: str) -> int:
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text: str) -> int:
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text: str) -> int:
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response: str, test_case: dict[str, Any]) -> int:
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

---
Remaining: **$6.36**

In [20]:
def run_prompt(test_case: dict[str, Any]) -> str:
    """Merges the prompt and test case input, then returns the result"""

    prompt = textwrap.dedent(f"""
        Please solve the following task:

        {test_case["task"]}
        * Respond only with Python, JSON, or a plain Regex
        * Do not add any comments or commentary or explanation
    """).strip()

    messages.clear()
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    return chat(messages, stop_sequences=["```"])


def run_test_case(test_case: dict[str, Any]) -> dict[str, Any]:
    """Calls run_prompt, then grades the result"""

    output = run_prompt(test_case)

    # Grade the output
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    syntax_score = grade_syntax(output, test_case)

    return {
        "output": output,
        "test_case": test_case,
        "score": (model_score + syntax_score) / 2.0,
        "reasoning": model_grade["reasoning"],
    }


def run_eval(dataset: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """Loads the dataset and calls run_test_case with each case"""

    results: list[dict[str, Any]] = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [ ]:
with open("dataset.json") as f:
    dataset = json.load(f)

results = run_eval(dataset)
with open("results.json", "w") as f:
    json.dump(results, f, indent=2)

remaining_credits()

Average score: 6.666666666666667


---
Remaining: **$6.36**

In [ ]:
remaining_credits()